<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 9 · Datos que se validan solos
**Excel Corporativo · 12 Claves** — Validación de datos y formato condicional · ⏱ 25 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/09_validacion_y_formato.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-09)

**Al terminar podrás:**
1. Ver cuántas formas distintas de escribir la misma ciudad llegan sin validación.
2. Medir cómo se reparte mal la venta cuando las variantes no se corrigen.
3. Usar una regla de formato condicional para que las excepciones salten a la vista.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · Seis ciudades, muchas formas de escribirlas
La columna `Ciudad` de la hoja **ERP_Sucio** se escribe a mano en cada sede. Sin una lista desplegable, cada persona la escribe a su manera: con o sin tilde, en mayúsculas, con un espacio al final o abreviada.

In [ ]:
# @title Leer la hoja ERP_Sucio y la tabla de equivalencias
import unicodedata
OFICIALES = sedes.ciudad.tolist()
ALIAS = {"bquilla": "Barranquilla", "santiago de cali": "Cali", "bogota dc": "Bogotá"}
def sin_tildes(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
def oficial(texto):
    clave = sin_tildes(str(texto)).lower().replace("/", "").replace(".", "").strip()
    for o in OFICIALES:
        if sin_tildes(o).lower() == clave:
            return o
    return ALIAS.get(clave, "Sin equivalencia")
erp = pd.read_excel(ruta("Distribuidora_Andina_practica.xlsx"), sheet_name="ERP_Sucio", dtype=object)
erp["Ciudad oficial"] = erp["Ciudad"].map(oficial)
erp["Valor"] = erp["Valor Neto"].map(lambda x: pd.to_numeric(x.replace("$", "").replace(".", "").strip()) if isinstance(x, str) else x).astype(float)
display(Markdown(f"Filas: **{miles(len(erp))}**. Formas distintas de escribir la ciudad: **{erp['Ciudad'].nunique()}**."))

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿De cuántas formas llega escrita cada ciudad? |
> | **Qué se grafica** | Barras: filas por cada forma escrita, coloreadas por si coinciden con la lista oficial |
> | **Por qué esta forma** | Ordenadas de mayor a menor, muestran que el problema no es raro: son cientos de filas. |
> | **Qué se descartó** | Una torta: con 16 porciones no se puede leer. |
> | **Cómo se lee** | Las barras azules son los nombres oficiales; las magenta son variantes que una lista desplegable habría impedido. |
> | **En Excel 365** | Formato condicional con fórmula =CONTAR.SI(Ciudades;$F2)=0 sobre la columna Ciudad. |

In [ ]:
# @title Gráfico 1 · Formas de escribir la ciudad
formas = erp["Ciudad"].value_counts().sort_values()
etiquetas = [f"«{c}»".replace(" »", "·»") for c in formas.index]
colores = [AZUL if c in OFICIALES else MAGENTA for c in formas.index]
fig = go.Figure(go.Bar(x=formas.values, y=etiquetas, orientation="h", marker_color=colores,
                       text=[miles(v) for v in formas.values], textposition="outside", cliponaxis=False))
variantes = formas[[c not in OFICIALES for c in formas.index]]
fig.update_layout(title=f"{len(formas)} formas de escribir {len(OFICIALES)} ciudades · {miles(variantes.sum())} filas con variantes",
                  xaxis_title="Filas", height=520)
dar_aire(fig, formas.max())
fig.show()
display(Markdown("El punto (·) antes de las comillas de cierre marca un espacio al final, invisible en Excel."))

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuánta venta se le asigna mal a cada ciudad sin corregir las variantes? |
> | **Qué se grafica** | Barras agrupadas por ciudad: venta con el nombre exacto frente a venta después de normalizar |
> | **Por qué esta forma** | La diferencia entre las dos barras de cada ciudad es la venta que una tabla dinámica pone en otra fila. |
> | **Qué se descartó** | Solo la venta normalizada: no se ve el tamaño del error. |
> | **Cómo se lee** | Donde la barra magenta es más corta, la ciudad aparece con menos venta de la real en un informe sin limpieza. |
> | **En Excel 365** | Tabla dinámica de ERP_Sucio con Ciudad en Filas: cuenta las filas que aparecen. |

In [ ]:
# @title Gráfico 2 · Venta por ciudad, con y sin normalizar
exacta = erp[erp["Ciudad"].isin(OFICIALES)].groupby("Ciudad")["Valor"].sum().reindex(OFICIALES) / 1e6
normal = erp.groupby("Ciudad oficial")["Valor"].sum().reindex(OFICIALES) / 1e6
fig = go.Figure()
fig.add_bar(name="Solo el nombre exacto", x=OFICIALES, y=exacta.values, marker_color=MAGENTA)
fig.add_bar(name="Después de normalizar", x=OFICIALES, y=normal.values, marker_color=AZUL)
perdida = 1 - exacta.sum() / normal.sum()
fig.update_layout(barmode="group", yaxis_title=MM, height=380, legend=dict(orientation="h", y=-0.15),
                  title=f"Sin normalizar, el {porc(perdida)} de la venta queda en filas de ciudades «fantasma»")
fig.show()

## 2 · Formato condicional: que la excepción salte a la vista
La validación evita el error al escribir. El formato condicional no impide nada: **resalta** lo que necesita atención. Funciona mejor cuando expresa una regla de negocio. En Distribuidora Andina, un descuento del 10 % o más requiere aprobación del jefe comercial.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué vendedores concentran los descuentos que requieren aprobación? |
> | **Qué se grafica** | Barras por vendedor: líneas de 2025 con descuento igual o mayor al límite · Resaltadas: las tres con más casos |
> | **Por qué esta forma** | Es lo que hace una regla de formato condicional: todo en gris y solo las excepciones con color. |
> | **Qué se descartó** | Todas las barras con color: la vista no sabe dónde detenerse. |
> | **Cómo se lee** | Las barras resaltadas son las primeras conversaciones del jefe comercial. |
> | **En Excel 365** | Regla con fórmula =$H2>=0,10 sobre la tabla de pedidos, con relleno amarillo para la fila completa. |

In [ ]:
# @title Gráfico 3 · Excepciones por vendedor
limite_descuento = 0.10  # @param [0.10, 0.15]
casos = (validas[(validas.anio == 2025) & (validas.descuento_pct >= limite_descuento)]
         .groupby("nombre_vendedor").size().sort_values())
top3 = casos.index[-3:]
fig = go.Figure(go.Bar(x=casos.values, y=casos.index, orientation="h",
                       marker_color=[MAGENTA if v in top3 else GRIS for v in casos.index],
                       text=[miles(v) for v in casos.values], textposition="outside", cliponaxis=False))
fig.update_layout(title=f"{miles(casos.sum())} líneas de 2025 con descuento ≥ {limite_descuento:.0%}: tres vendedores concentran el {porc(casos[top3].sum() / casos.sum())}".replace("%", " %", 1),
                  xaxis_title="Líneas", height=520)
dar_aire(fig, casos.max())
fig.show()

## 3 · Retos de interpretación

In [ ]:
# @title Reto 1 · ¿De cuántas formas distintas llega escrita la ciudad?
respuesta = "Elige una opción"  # @param ["Elige una opción", "6", "10", "16", "20"]
revisar(respuesta, str(erp["Ciudad"].nunique()), """Es el número de barras del gráfico 1. Solo seis son correctas.""")

In [ ]:
# @title Reto 2 · Agregas una lista desplegable a la columna Ciudad. ¿Qué pasa con las variantes que ya estaban escritas?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Se corrigen solas", "Siguen ahí: la validación solo controla lo que se escribe después", "Se borran", "Excel muestra un error en cada una"]
revisar(respuesta, "Siguen ahí: la validación solo controla lo que se escribe después", """Para encontrarlas: Datos › Validación de datos › Rodear con un círculo datos no válidos.""")

In [ ]:
# @title Reto 3 · ¿Qué fórmula de formato condicional resalta la fila completa cuando el descuento (columna H) es del 10 % o más?
respuesta = "Elige una opción"  # @param ["Elige una opción", "=H2>=0,10", "=$H2>=0,10", "=$H$2>=0,10", "=H$2>=0,10"]
revisar(respuesta, "=$H2>=0,10", """El $ fija la columna H; la fila queda libre para que cada fila se evalúe con su propio descuento.""")

## Apóyate en IA
Para diseñar las reglas de un formato antes de entregarlo:

```text
Voy a crear en Excel 365 un formato de pedidos que llenan 18 vendedores (fecha, vendedor, ciudad, canal, producto, cantidad, precio, descuento).
Propón para cada columna una regla de validación de datos y el mensaje de error, y dos reglas de formato condicional con fórmula para resaltar excepciones de negocio.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Una lista desplegable evita las variantes; no corrige las que ya existen.
* Las listas que apuntan a una Tabla crecen solas.
* El formato condicional sirve cuando resalta pocas cosas y con un motivo de negocio.

**Siguiente:** [Clave 10 · Fórmulas de matriz dinámica](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-10) · [Laboratorio](https://omarli-360.github.io/excel-corporativo-12-claves/#laboratorio)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>